categorias da COS presentes na zona de interesse

In [ ]:
# ==============================================================================
# SCRIPT MÍNIMO: VISUALIZAÇÃO DA COS E CATEGORIAS
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np # Necessário para funções de array
import os # Necessário para verificar arquivos (embora não usado diretamente no plot)

# --- 1. DEFINIÇÃO DE COORDENADAS E ARQUIVOS (SETUP) ---
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"

# Limites da área de interesse (em EPSG:3035)
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

# --- 2. CARREGAMENTO E REPROJEÇÃO ---
try:
    # Carrega a COS e reprojeta para o Web Mercator (EPSG:3857, padrão para contextily)
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    cos_3857 = cos.to_crs(epsg=3857)
    
    # Cria uma caixa de recorte (no CRS de destino)
    clip_box_3035 = box(este_min, norte_min, este_max, norte_max)
    clip_box_3857 = gpd.GeoSeries(clip_box_3035, crs='EPSG:3035').to_crs(epsg=3857).iloc[0]
    
    # Calcula os limites de zoom a partir da caixa
    minx, miny, maxx, maxy = clip_box_3857.bounds
    pad = (maxx - minx) * 0.05
    x0, x1 = minx - pad, maxx + pad
    y0, y1 = miny - pad, maxy + pad

    # Recorta a COS para a área de interesse
    cos_clip = cos_3857.clip(clip_box_3857)

except Exception as e:
    print(f"ERRO: Não foi possível carregar a COS ou calcular limites. Verifique o COS_PATH. {e}")
    exit()

# --- 3. ANÁLISE: IMPRIMIR CATEGORIAS PRESENTES ---
cos_categories = cos_clip['COS23_n4_L'].unique()
print("\n=======================================================")
print("  CATEGORIAS COS (COS23_n4_L) PRESENTES NA ÁREA")
print("=======================================================")
for category in sorted(cos_categories):
    print(f"- {category}")
print("-------------------------------------------------------")


# ============================================================
# FIGURA 1: VISUALIZAÇÃO SIMPLES DA COS
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plotar a COS recortada, usando a coluna de categorias para a cor
cos_clip.plot(
    column='COS23_n4_L', 
    ax=ax, 
    cmap='tab20', # Mapa de cores diversificado
    alpha=0.7, 
    edgecolor='k', 
    linewidth=0.2, 
    legend=True, 
    legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1), 'fontsize': 8}
)

# Adicionar Mapa Base para Contexto
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs=cos_3857.crs.to_string())

# Formatação Final
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)
ax.set_title("Carta de Uso e Ocupação do Solo (COS) 2023 - Área Recortada", fontsize=14)
ax.set_axis_off()

plt.tight_layout()
plt.show()

# Deslocamento vertical, dV

## Temperatura

### K=2

todos os pontos das celulas quadradas

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados (para acelerar o processamento)
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha sobre asc_interp
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Criar colunas cell_x, cell_y, cell_id
asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Criar GeoDataFrame da grelha
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Recorte à barragem
grid_barragem = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

# ==============================
# 7. Clustering apenas células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam a barragem
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 8. Plot mapa + séries temporais
# ==============================
# Pontos centrais
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Plot
fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

#gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

# Carregar temperatura
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Eixos
    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


todos os pontos, agregações, clustering e plots sejam feitos apenas com os pontos dentro da barragem, incluindo o overlay da temperatura nas séries temporais.

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (VERSÃO FINAL CORRIGIDA)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites da barragem
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')


# =====================================================================
# 9. Leitura/Criação de dados hidro-climáticos
# =====================================================================
date_range = agg_pivot.columns

# --- 9.1 Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    print("AVISO: Ficheiro de temperatura não encontrado. Usando dados placeholder.")
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10 * np.sin(np.pi * 2 * (date_range - date_range.min()).days / 365) + np.random.randn(len(date_range)) * 2
    })
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])

temp_series = df_temp_raw.set_index('data')['med']
temp_aligned = temp_series.reindex(date_range, fill_value=np.nan)

df_temp = pd.DataFrame({
    'data': temp_aligned.index,
    'med': temp_aligned.values
})
df_temp['data'] = pd.to_datetime(df_temp['data'])

window = 13
if len(df_temp) < window: window = len(df_temp) if len(df_temp) % 2 != 0 else len(df_temp) - 1
if window < 3: window = 3
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)


# ==============================
# 9.2 Nível da água
# ==============================
window_smooth = 13
if len(date_range) < window_smooth:
    window_smooth = len(date_range) if len(date_range) % 2 != 0 else len(date_range) - 1
if window_smooth < 3:
    window_smooth = 3

try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    print("AVISO: Ficheiro de Nível não encontrado. Usando dados placeholder.")
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })

df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range)
nivel_aligned = nivel_series.interpolate(method='time').ffill().bfill()

df_nivel = pd.DataFrame({
    'data': nivel_aligned.index,
    'nivel': nivel_aligned.values
})
df_nivel['data'] = pd.to_datetime(df_nivel['data'])

df_nivel['nivel_smooth'] = savgol_filter(
    df_nivel['nivel'],
    window_length=window_smooth,
    polyorder=2
)


# --- 9.3 Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    print("AVISO: Ficheiro de Precipitação não encontrado. Usando dados placeholder.")
    np.random.seed(42)
    prec_values = np.clip(np.random.normal(loc=50, scale=30, size=len(date_range)), a_min=0, a_max=None)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': prec_values})

prec_series = df_prec_raw.set_index('data')['prec']
prec_aligned = prec_series.reindex(date_range, fill_value=0)

df_prec = pd.DataFrame({
    'data': prec_aligned.index,
    'prec': prec_aligned.values
})

df_prec['prec_acum'] = df_prec['prec'].cumsum()

def get_hydro_year(date):
    return date.year if date.month >= 10 else date.year - 1

df_prec['data'] = pd.to_datetime(df_prec['data']) 
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year) 
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()


# ==============================
# 10. Funções de Escalonamento Visual
# ==============================
# Offset aumentado para 0.30 para baixar as séries do topo
# Scale reduzido para 0.20 para compactar ligeiramente
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    
    if dmax == dmin:
        norm = np.zeros_like(data)
        dmin, dmax = 0, 1
    else:
        norm = (data - dmin) / (dmax - dmin)
    
    visual = norm * dV_span * scale_factor
    visual = visual + (dV_max - dV_span * offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real:
        real_ticks = np.linspace(d_min_real - 0.5, d_max_real + 0.5, num_ticks)
    else:
        real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)

    dV_span = dV_max - dV_min
    if d_max_real == d_min_real:
        norm_ticks = np.linspace(0, 1, num_ticks)
    else:
        norm_ticks = (real_ticks - d_min_real) / (d_max_real - d_min_real)

    visual_ticks = norm_ticks * dV_span * scale_factor + (dV_max - dV_span * offset_factor)
    return visual_ticks, real_ticks

# ==============================
# 11. Plot Mapa e Múltiplas Séries Temporais
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig = plt.figure(figsize=(5.5 * n_clusters, 12 + 3.0 * n_rows_series)) 
gs = fig.add_gridspec(n_rows_series + 1, n_clusters, height_ratios=[12] + [3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- LINHA 1: MAPA ---
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points, geometry=gpd.points_from_xy(points['easting'], points['northing']), crs="EPSG:3035"
).to_crs(epsg=3857)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]
legend_handles_map.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, label='Pontos InSAR (Média da Célula)'))

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Clusters de Deslocamento Vertical (dV) na Barragem\nBaseado em Séries Temporais (2019-2023)", fontsize=14)

# --- Séries visuais (Offset 0.30 para baixar do topo) ---
temp_visual, temp_min_real, temp_max_real = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min_real, nivel_max_real = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, prec_total_acum_min_real, prec_total_acum_max_real = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig.add_subplot(gs[row_idx + 1, idx])
        
        # Plot dV
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5, label=f'Média Cluster {cluster_id+1}')
        
        # Plot Eixo Secundário
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            # COMBINADO: BARRAS TEAL + LINHAS TEAL
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3, label='Precipitação (mm)')
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=2, label=f'Acumulado {ano}', alpha=0.9)
            
            prec_max_real = df_prec['prec_acum_anual'].max()
            ax2.set_ylim(0, prec_max_real * 1.1)
            ax2.tick_params(axis='y', labelcolor='teal')
            ext_color_label = 'teal'
            
            lines2 = [
                plt.Rectangle((0, 0), 1, 1, fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0], [0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]
            labels2 = [h.get_label() for h in lines2]

        elif plot_as_bar:
            # BARRAS SIMPLES (TEAL)
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5, label=ext_label)
            d_max_real = ext_data_df[ext_col_real].max()
            ax2.set_ylim(0, d_max_real * 1.1)
            
            lines2 = [plt.Rectangle((0, 0), 1, 1, fc=bar_color or ext_color, alpha=0.5)]
            labels2 = [ext_label]
            ext_color_label = bar_color or ext_color
            ax2.tick_params(axis='y', labelcolor=ext_color_label)

        elif plot_real_scale_line:
            # LINHA ESCALA REAL + BARRAS FUNDO (TEAL + NAVY -> Pedido: TEAL)
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3, label='Precipitação mensal') 
            
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=2.2, alpha=0.85, label=ext_label)
            
            d_max_real = ext_data_df[ext_col_real].max()
            ax2.set_ylim(0, d_max_real * 1.1)
            
            lines2 = [
                plt.Rectangle((0, 0), 1, 1, fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0], [0], color=ext_color, linewidth=2, label=ext_label)
            ]
            labels2 = [h.get_label() for h in lines2]
            
            ext_color_label = ext_color 
            ax2.tick_params(axis='y', labelcolor=ext_color_label)
        
        else: 
            # LINHA ESCALONADA (TEMP, NÍVEL) - Offset 0.30
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=2.2, alpha=0.85, label=ext_label)

            valid_data = ext_data_df[ext_col_real].dropna()
            if valid_data.empty:
                d_min_real, d_max_real = 0, 1
            else:
                d_min_real, d_max_real = valid_data.min(), valid_data.max()
            
            visual_ticks, real_ticks = create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
            
            ax2.set_ylim(dV_ylim) 
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax2.tick_params(axis='y', labelcolor=ext_color_label)

        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])
        
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "")
            ax2.set_ylabel(f'{clean_label}', color=ext_color_label)
        else:
            ax2.set_yticklabels([])
        
        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else: ax.set_xticklabels([])
            
        ax.grid(False)
        ax2.grid(False)

        # Legenda: Fundo Opaque, Z-Order Alto
        lines1, labels1 = ax.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='best', framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)
        
        return ax

    # 2: Temp (Preto, Offset 0.30)
    plot_dV_comparison(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    # 3: Nível (Navy, Offset 0.30)
    plot_dV_comparison(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    # 4: Prec Mensal (Teal)
    plot_dV_comparison(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    # 5: Prec Acum Total (Teal Linha + Teal Barras) -> Pedido: TEAL
    plot_dV_comparison(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    # 6: Prec Acum Anual (Teal Linha + Teal Barras) -> Pedido: TEAL
    plot_dV_comparison(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# FIGURA 2: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
# figsize = (largura, altura)
fig_series, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            # Period=12 para dados mensais
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem (Sem Grid, Altura Reduzida) ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=0.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False) # Grid removida

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False) # Grid removida

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False) # Grid removida

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False) # Grid removida

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA
# ==============================
fig_map = plt.figure(figsize=(5.5 * n_clusters, 12))
gs_map = fig_map.add_gridspec(1, n_clusters)

ax_map = fig_map.add_subplot(gs_map[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points, geometry=gpd.points_from_xy(points['easting'], points['northing']), crs="EPSG:3035"
).to_crs(epsg=3857)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]
legend_handles_map.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, label='Pontos InSAR (Média da Célula)'))

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Clusters de Deslocamento Vertical (dV) na Barragem\nBaseado em Séries Temporais (2019-2023)", fontsize=14)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================
fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    # --- Função de plotagem reutilizando temp_visual, nivel_visual e prec_total_acum_visual ---
    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color,
                                plot_as_bar=False, bar_color=None, plot_real_scale_line=False,
                                combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'], color='teal', linewidth=2, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=2.2, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color=ext_color, linewidth=2, label=ext_label)
            ]

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=2.2, alpha=0.85, label=ext_label)
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]  # último plot como legenda

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])

        # y2 label somente no último cluster
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "")
            ax2.set_ylabel(f'{clean_label}', color=ext_color if ext_color != 'black' else 'dimgray')
        else:
            ax2.set_yticklabels([])

        # x-axis
        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else: ax.set_xticklabels([])

        # --- Legenda combinada ---
        lines1, labels1 = ax.get_legend_handles_labels()
        labels2 = [l.get_label() for l in lines2]
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='best', framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax


    # 0: Temp (preto)
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    # 1: Nível (navy)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    # 2: Precip Mensal (teal)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    # 3: Precip Acum Total (teal)
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    # 4: Precip Acum Anual (teal)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=0.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================
fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    # --- Função de plotagem reutilizando temp_visual, nivel_visual e prec_total_acum_visual ---
    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color,
                                plot_as_bar=False, bar_color=None, plot_real_scale_line=False,
                                combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color=ext_color, linewidth=2, label=ext_label)
            ]

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]  # último plot como legenda

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])

        # y2 label somente no último cluster
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "")
            ax2.set_ylabel(f'{clean_label}', color=ext_color if ext_color != 'black' else 'dimgray')
        else:
            ax2.set_yticklabels([])

        # x-axis
        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else: ax.set_xticklabels([])

        # --- Legenda combinada ---
        lines1, labels1 = ax.get_legend_handles_labels()
        labels2 = [l.get_label() for l in lines2]
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='best', framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax


    # 0: Temp (preto)
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    # 1: Nível (navy)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    # 2: Precip Mensal (teal)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    # 3: Precip Acum Total (teal)
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    # 4: Precip Acum Anual (teal)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (CORRIGIDO)
# ==============================
print("Gerando Figura 2...")
clusters_present = sorted(cluster_df['cluster'].unique())
nc = len(clusters_present)
nr = 5

fig_series = plt.figure(figsize=(5.5 * nc, 3.0 * nr))
gs_series = fig_series.add_gridspec(nr, nc, height_ratios=[3.0] * nr)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# --- CORREÇÃO AQUI: Nomes das variáveis alinhados com as chamadas ---
# Temperatura
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max)
# Nível
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max)
# Prec. Acumulada (para escala visual, caso seja necessário)
prec_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # Plot dV
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.5, label=f'Média Cluster {cluster_id+1}')
        
        # Plot Eixo Secundário
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec: # Prec. Anual
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5)
            
            # Escala real para acumulado anual
            prec_max = df_prec['prec_acum_anual'].max()
            if pd.isna(prec_max) or prec_max == 0: prec_max = 1.0
            ax2.set_ylim(0, prec_max * 1.1)
            
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal'), Line2D([0],[0], color='teal', linewidth=1.5, label='Prec. anual')]
            ext_color_label = 'teal'

        elif plot_as_bar: # Prec. Mensal Simples
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            
            vmax = ext_data_df[ext_col_real].max()
            if pd.isna(vmax) or vmax == 0: vmax = 1.0
            ax2.set_ylim(0, vmax * 1.1)
            
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line: # Prec. Acumulada Total (Escala Real)
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            
            vmax = ext_data_df[ext_col_real].max()
            if pd.isna(vmax) or vmax == 0: vmax = 1.0
            ax2.set_ylim(0, vmax * 1.1)
            
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal'), Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color 

        else: # Escalonado (Temp/Nível)
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            
            dmin_r, dmax_r = ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max()
            if pd.isna(dmin_r): dmin_r=0; dmax_r=1
            
            vt, rt = create_visual_ticks(dmin_r, dmax_r, dV_min, dV_max)
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(vt)
            
            fmt = "{:.0f}" if integer_ticks else "{:.1f}"
            ax2.set_yticklabels([fmt.format(x) for x in rt])
            
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configurações Finais ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1}', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])

        if idx == n_clusters - 1:
            ax2.set_ylabel(ext_label if not combine_annual_prec else 'mm', color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label)
        else:
            ax2.set_yticklabels([])

        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
        else: ax.set_xticklabels([])

        ax.grid(False); ax2.grid(False)
        
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='best', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # --- Chamadas das Funções ---
    # 0: Temp (preto) - Passamos temp_visual
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temp. Média (°C)', 'black', integer_ticks=True)
    
    # 1: Nível (navy) - Passamos nivel_visual
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível (m)', 'navy', integer_ticks=True)
    
    # 2: Precip Mensal (teal)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Prec. Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    
    # 3: Prec Acum Total (teal) - Escala Real
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    
    # 4: Prec Acum Anual (teal)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. Anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (CORRIGIDO)
# ==============================
print("Gerando Figura 2...")
clusters_present = sorted(cluster_df['cluster'].unique())
nc = len(clusters_present)
nr = 5

fig_series = plt.figure(figsize=(5.5 * nc, 3.0 * nr))
gs_series = fig_series.add_gridspec(nr, nc, height_ratios=[3.0] * nr)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# --- CORREÇÃO AQUI: Nomes das variáveis alinhados com as chamadas ---
# Temperatura
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max)
# Nível
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max)
# Prec. Acumulada (para escala visual, caso seja necessário)
prec_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # Plot dV
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.5, label=f'Média Cluster {cluster_id+1}')
        
        # Plot Eixo Secundário
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec: # Prec. Anual
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5)
            
            # Escala real para acumulado anual
            prec_max = df_prec['prec_acum_anual'].max()
            if pd.isna(prec_max) or prec_max == 0: prec_max = 1.0
            ax2.set_ylim(0, prec_max * 1.1)
            
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal'), Line2D([0],[0], color='teal', linewidth=1.5, label='Prec. anual')]
            ext_color_label = 'teal'

        elif plot_as_bar: # Prec. Mensal Simples
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            
            vmax = ext_data_df[ext_col_real].max()
            if pd.isna(vmax) or vmax == 0: vmax = 1.0
            ax2.set_ylim(0, vmax * 1.1)
            
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line: # Prec. Acumulada Total (Escala Real)
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            
            vmax = ext_data_df[ext_col_real].max()
            if pd.isna(vmax) or vmax == 0: vmax = 1.0
            ax2.set_ylim(0, vmax * 1.1)
            
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal'), Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color 

        else: # Escalonado (Temp/Nível)
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            
            dmin_r, dmax_r = ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max()
            if pd.isna(dmin_r): dmin_r=0; dmax_r=1
            
            vt, rt = create_visual_ticks(dmin_r, dmax_r, dV_min, dV_max)
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(vt)
            
            fmt = "{:.0f}" if integer_ticks else "{:.1f}"
            ax2.set_yticklabels([fmt.format(x) for x in rt])
            
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configurações Finais ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1}', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])

        if idx == n_clusters - 1:
            ax2.set_ylabel(ext_label if not combine_annual_prec else 'mm', color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label)
        else:
            ax2.set_yticklabels([])

        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
        else: ax.set_xticklabels([])

        ax.grid(False); ax2.grid(False)
        
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='best', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # --- Chamadas das Funções ---
    # 0: Temp (preto) - Passamos temp_visual
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temp. Média (°C)', 'black', integer_ticks=True)
    
    # 1: Nível (navy) - Passamos nivel_visual
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível (m)', 'navy', integer_ticks=True)
    
    # 2: Precip Mensal (teal)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Prec. Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    
    # 3: Prec Acum Total (teal) - Escala Real
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    
    # 4: Prec Acum Anual (teal)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. Anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 4
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)


# --- Séries visuais (Offset 0.30 para baixar do topo) ---
temp_visual, temp_min_real, temp_max_real = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min_real, nivel_max_real = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, prec_total_acum_min_real, prec_total_acum_max_real = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    # --- Função de plotagem reutilizando temp_visual, nivel_visual e prec_total_acum_visual ---
    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color,
                                plot_as_bar=False, bar_color=None, plot_real_scale_line=False,
                                combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color=ext_color, linewidth=2, label=ext_label)
            ]

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]  # último plot como legenda

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])

        # y2 label somente no último cluster
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "")
            ax2.set_ylabel(f'{clean_label}', color=ext_color if ext_color != 'black' else 'dimgray')
        else:
            ax2.set_yticklabels([])

        # x-axis
        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else: ax.set_xticklabels([])

        # --- Legenda combinada ---
        lines1, labels1 = ax.get_legend_handles_labels()
        labels2 = [l.get_label() for l in lines2]
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='best', framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax


    # 0: Temp (preto)
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    # 1: Nível (navy)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    # 2: Precip Mensal (teal)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    # 3: Precip Acum Total (teal)
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    # 4: Precip Acum Anual (teal)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 25
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)


# --- Séries visuais (Offset 0.30 para baixar do topo) ---
temp_visual, temp_min_real, temp_max_real = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min_real, nivel_max_real = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, prec_total_acum_min_real, prec_total_acum_max_real = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    # --- Função de plotagem reutilizando temp_visual, nivel_visual e prec_total_acum_visual ---
    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color,
                                plot_as_bar=False, bar_color=None, plot_real_scale_line=False,
                                combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color=ext_color, linewidth=2, label=ext_label)
            ]

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]  # último plot como legenda

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: ax.set_ylabel('dV (mm)')
        else: ax.set_yticklabels([])

        # y2 label somente no último cluster
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "")
            ax2.set_ylabel(f'{clean_label}', color=ext_color if ext_color != 'black' else 'dimgray')
        else:
            ax2.set_yticklabels([])

        # x-axis
        if row_idx == n_rows_series - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else: ax.set_xticklabels([])

        # --- Legenda combinada ---
        lines1, labels1 = ax.get_legend_handles_labels()
        labels2 = [l.get_label() for l in lines2]
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='best', framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax


    # 0: Temp (preto)
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    # 1: Nível (navy)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    # 2: Precip Mensal (teal)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    # 3: Precip Acum Total (teal)
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    # 4: Precip Acum Anual (teal)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11, fontweight='bold')
    if idx == 0: ax0.set_ylabel('Observed', fontsize=9)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=9)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=9)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=9)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)

            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color='teal', linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)

            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            # --- Tratamento especial: Temperatura (preto) ---
            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        # --- Remover ticks exteriores e deixar somente extremos ---
        ax.tick_params(left=False)
        ax2.tick_params(right=False)

        if idx == 0:
            ax.tick_params(left=True)

        if idx == n_clusters - 1:
            ax2.tick_params(right=True)

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda ---
        lines1, labels1 = ax.get_legend_handles_labels()
        labels2 = [l.get_label() for l in lines2]

        ax.legend(
            lines1 + lines2,
            labels1 + labels2,
            fontsize=8,
            loc='best',
            framealpha=1.0,
            facecolor='white',
            edgecolor='lightgray'
        ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex='col')

# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)

            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color='teal', linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)

            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=2, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            # --- Tratamento especial: Temperatura (preto) ---
            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        # --- Remover ticks exteriores e deixar somente extremos ---
        ax.tick_params(left=False)
        ax2.tick_params(right=False)

        if idx == 0:
            ax.tick_params(left=True)

        if idx == n_clusters - 1:
            ax2.tick_params(right=True)

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])
        
        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            labels2 = [l.get_label() for l in lines2]

            ax.legend(
                lines1 + lines2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (CORRIGIDO: Y2 APENAS NA ÚLTIMA COLUNA)
# ==============================
print("Gerando Figura 2...")
clusters_present = sorted(cluster_df['cluster'].unique())
nc = len(clusters_present)
nr = 5

fig_series = plt.figure(figsize=(5.5 * nc, 3.0 * nr))
gs_series = fig_series.add_gridspec(nr, nc, height_ratios=[3.0] * nr)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max)
nv, nmn, nmx = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max)
ptv, ptmi, ptmx = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max)

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # Plot dV
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # Plot Eixo Secundário
        ax2 = ax.twinx()
        lines2, labels2 = [], []
        ext_color_label = ext_color
        
        # Lógica de plotagem (igual à anterior)
        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', lw=2)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'), Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')]
            ext_color_label = 'teal'

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars: ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'), Line2D([0],[0], color=ext_color, linewidth=2, label=ext_label)]
            ext_color_label = ext_color 

        else: # Escalonado
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            vt, rt = create_visual_ticks(ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), dV_min, dV_max)
            ax2.set_ylim(dV_ylim); ax2.set_yticks(vt)
            
            # Só definir labels dos ticks se for a última coluna
            if idx == nc - 1:
                fmt = "{:.0f}" if integer_ticks else "{:.1f}"
                ax2.set_yticklabels([fmt.format(x) for x in rt])
            else:
                ax2.set_yticklabels([]) # Ticks vazios para outras colunas
            
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configuração Final dos Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        
        # Y1 (Esquerda): Só na primeira coluna
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False) # Remove ticks visuais

        # Y2 (Direita): Só na última coluna
        if idx == nc - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False) # Remove ticks visuais

        # X-Axis
        if row_idx == nr - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
        else: ax.set_xticklabels([])

        ax.grid(False); ax2.grid(False)
        
        # Legenda (melhorada)
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='best', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # Chamadas
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 5
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (X-AXIS CORRIGIDO)
# ==============================
print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Prec. acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            visual_ticks, real_ticks = create_visual_ticks(ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
            ax2.set_ylim(dV_ylim); ax2.set_yticks(visual_ticks)
            if integer_ticks: ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else: ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configuração Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        
        # Y1 (Esquerda)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 (Direita)
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # --- Eixo X (CORREÇÃO AQUI) ---
        if row_idx == n_rows_series - 1:
            # Última linha: Formato 2023, 2024 e Ticks visíveis
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else:
            # Linhas superiores: Sem texto e sem ticks
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # Chamadas
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

dH

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
#agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dH','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# --- Séries visuais ---
#temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
#nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
#prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Equipamentos culturais']
    barragem = cos[cos['COS23_n4_L'].isin(['Infraestruturas de produção de energia hídrica', 'Equipamentos culturais'])]

except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 25
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### K=3

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 25
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### K=4

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=5

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# Mantendo células com pontos INSAR mesmo que o centroid fique fora
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_barragem_recort = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)

valid_cell_ids = set()
for pt in points_gdf.geometry:
    # query retorna os índices dos polígonos que podem conter/intersectar o ponto
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])

grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]


# ==============================
# 8. Clustering das células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam barragem com pontos
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 5
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Plot mapa + séries temporais com temperatura
# ==============================
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Nível da albufeira

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar temperatura e nível
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais com nível ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

nivel_min = df_nivel['nivel_smooth'].min()
nivel_max = df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar temperatura e nível
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais com nível ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

nivel_min = df_nivel['nivel_smooth'].min()
nivel_max = df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries dV
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação (barras + linha média) ---
    ax2 = ax.twinx()

    # Barras de precipitação
    bar_container = ax2.bar(
        df_prec['data'], df_prec['prec'],
        width=20, color='royalblue', alpha=0.35, label='Precipitação (mm)'
    )

    # Coordenadas dos centros das barras e respetivos valores
    bar_centers = [bar.get_x() + bar.get_width()/2 for bar in bar_container]
    bar_heights = [bar.get_height() for bar in bar_container]

    # Linha conectando os pontos médios das barras
    ax2.plot(bar_centers, bar_heights, color='navy', linewidth=2.0, label='Tendência da precipitação')

    # Eixos e rótulos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Eixos e estilo ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=4

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=5

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 5
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total acumulada

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa de Clusters de dV com Centro de Células ASC/DESC", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total anual acumulada

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: Mapa
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(ax=ax_map,
                                                               color=cluster_colors[int(row['cluster'])],
                                                               alpha=0.4)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    ax_map.scatter([], [], color=cluster_colors[cluster_id], alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa de Clusters de dV com Centro de Células ASC/DESC", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()